In [0]:
from pyspark.sql import functions as F, Window

bronze = spark.table("workspace.bronze.cotacoes_raw")
print("linhas na bronze:", bronze.count())   # esperado: 30294 linhas

# Remover colunas vazias ou irrelevantes que não carregam informação de negócio
RUIDO = ["codigo_bdi", "tipo_mercado", "moeda", "fator_cotacao", "prazo_dias_termo", "codigo_isin"]
df = bronze.drop(*RUIDO)
print(len(bronze.columns), "->", len(df.columns), "colunas") 

display(df)

In [0]:
#tirar espaços desnecessários e transformar textos que representam ausência de informação em NULL de verdade
NULOS_DISFARCADOS = ["", "N/A", "N/D", "NULL"]
def limpa_texto(nome_coluna):
    t = F.trim(F.col(nome_coluna))
    return F.when(F.upper(t).isin(NULOS_DISFARCADOS), None).otherwise(t)

df = df.withColumn("ticker", F.upper(F.col("ticker")))

for c in df.columns:
    if not c.startswith("_"):
        df = df.withColumn(c, limpa_texto(c))

#Criei uma função reutilizável de limpeza textual que aplica trim e converte representações textuais de ausência de informação, como N/A, N/D e NULL, em valores nulos reais. Depois aplico essa regra de forma programática às colunas do DataFrame, evitando duplicação de código


In [0]:
# Padroniza id_empresa como inteiro para manter consistência com a Silver de empresas e garantir o JOIN na Gold
df = df.withColumn(
    "id_empresa",
    F.expr("try_cast(id_empresa as int)")
)

# Valida a conversão: tipo integer e nenhum valor perdido
df.printSchema()
print("id_empresa nulos:", df.filter(F.col("id_empresa").isNull()).count())

In [0]:
# Tratar coluna data_pregao
# Converte diferentes formatos de data para o tipo date
def converte_data(col):
    return F.coalesce(
        F.expr(f"try_to_date({col}, 'yyyy-MM-dd')"),
        F.expr(f"try_to_date({col}, 'dd/MM/yyyy')"),
        F.expr(f"try_to_date({col}, 'yyyyMMdd')")
    )

# Aplica a conversão na coluna data_pregao
df = df.withColumn("data_pregao", converte_data("data_pregao"))

In [0]:
# Valida tipo, nulos e intervalo das datas
df.printSchema()

print(
    "datas nulas:",
    df.filter(F.col("data_pregao").isNull()).count()
)

df.agg(
    F.min("data_pregao").alias("data_minima"),
    F.max("data_pregao").alias("data_maxima")
).show()

In [0]:
precos = ["preco_abertura", "preco_maximo", "preco_minimo", "preco_fechamento", "volume_financeiro"]
for c in precos:
    print(c, "| com vírgula:", df.filter(F.col(c).contains(",")).count(),
          "| negativos:", df.filter(F.col(c).startswith("-")).count(),
          "| nulos:", df.filter(F.col(c).isNull()).count())

In [0]:
# Para cada coluna de preços e volume, substitui vírgulas por pontos e converte para double (que utiliza ponto como separador decimal)
for c in precos:
    df = df.withColumn(
        c,
        F.expr(f"try_cast(replace({c}, ',', '.') as double)")
    )

In [0]:
# Validar a conversão para double
for c in precos:
    print(
        c,
        "| nulos:", df.filter(F.col(c).isNull()).count(),
        "| <= 0:", df.filter(F.col(c) <= 0).count()
    )

In [0]:
display(df)

In [0]:
# Converter qtd_negocios e qtd_titulos de texto para números inteiros (bigint), utilizando uma tipagem capaz de armazenar valores muito grandes
for c in ["qtd_negocios", "qtd_titulos"]:
    df = df.withColumn(c, F.col(c).try_cast("bigint"))

In [0]:
df.select("qtd_negocios", "qtd_titulos").printSchema()   # esperado: long (bigint)
for c in ["qtd_negocios", "qtd_titulos"]:
    print(c, "| nulos:", df.filter(F.col(c).isNull()).count(),
          "| <= 0:", df.filter(F.col(c) <= 0).count())   # esperado: 0 e 0 (todos os valores foram convertidos)

In [0]:
# Tratar registros duplicados por empresa e data, mantendo a cotação com ingestão mais recente
w = Window.partitionBy("id_empresa", "data_pregao").orderBy(F.col("_ingestion_ts").desc())
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")



In [0]:
# Validar o total de registros, empresas, dias de pregão e ausência de registros repetidos por empresa/data
print("linhas:", df.count())                                              
print("empresas:", df.select("id_empresa").distinct().count())            
print("dias:", df.select("data_pregao").distinct().count())               
print("duplicatas restantes:",
      df.groupBy("id_empresa", "data_pregao").count().filter("count > 1").count())   

In [0]:
# Verifica a qualidade dos preços e identifica o motivo de cada registro inválido
# Se houver algum problema, preenche motivo_rejeicao; se estiver tudo certo, fica NULL
# Regras: fechamento nulo ou <= 0, preços <= 0, máximo < mínimo ou fechamento fora do intervalo

motivo = (
    F.when(F.col("preco_fechamento").isNull(), "fechamento nulo")
     .when(F.col("preco_fechamento") <= 0, "fechamento <= 0")
     .when((F.col("preco_abertura") <= 0) | (F.col("preco_maximo") <= 0) | (F.col("preco_minimo") <= 0),
           "abertura/maximo/minimo <= 0")
     .when(F.col("preco_maximo") < F.col("preco_minimo"), "maximo < minimo")
     .when((F.col("preco_fechamento") > F.col("preco_maximo")) | (F.col("preco_fechamento") < F.col("preco_minimo")),
           "fechamento fora do intervalo")
)

df = df.withColumn("motivo_rejeicao", motivo)

# Separa os registros:
# cotacoes = registros válidos (motivo_rejeicao = NULL)
# quarentena = registros inválidos, mantendo o motivo para investigar depois

cotacoes = df.filter(F.col("motivo_rejeicao").isNull()).drop("motivo_rejeicao")
quarentena = df.filter(F.col("motivo_rejeicao").isNotNull())

In [0]:
n_ok, n_q = cotacoes.count(), quarentena.count()
print("cotações:", n_ok, "| quarentena:", n_q, "| soma:", n_ok + n_q)   # esperado: soma = 29700
quarentena.groupBy("motivo_rejeicao").count().show(truncate=False)

In [0]:
empresas = spark.table("workspace.silver.empresas")

# ANTES (bronze, tudo texto): '000123' != '123', então o JOIN não encontra nada
b_cot = spark.table("workspace.bronze.cotacoes_raw")
b_emp = spark.table("workspace.bronze.empresas_raw").select("id_empresa")
print("JOIN na bronze:", b_cot.join(b_emp, "id_empresa").count())                       # esperado: 0

# DEPOIS (silver, id_empresa int dos dois lados)
print("ids sem empresa:", cotacoes.join(empresas, "id_empresa", "left_anti").count())   # esperado: 0
print("linhas no JOIN:", cotacoes.join(empresas.select("id_empresa"), "id_empresa").count(),
      "| linhas em cotacoes:", n_ok)                                                    # esperado: iguais

# O ticker da cotação tem que ser o mesmo do cadastro
chk = cotacoes.join(empresas.select("id_empresa", F.col("ticker").alias("ticker_cadastro")), "id_empresa")
print("tickers divergentes:", chk.filter(F.col("ticker") != F.col("ticker_cadastro")).count())   # esperado: 0

In [0]:
# Sem partição: com ~30 mil linhas, particionar criaria muitos arquivos minúsculos (small files problem).
# Em volume grande eu particionaria por data.
cotacoes.write.mode("overwrite").saveAsTable("workspace.silver.cotacoes")
quarentena.write.mode("overwrite").saveAsTable("workspace.silver.cotacoes_quarentena")

In [0]:
c = spark.table("workspace.silver.cotacoes").count()
q = spark.table("workspace.silver.cotacoes_quarentena").count()
print("cotacoes:", c, "| quarentena:", q, "| soma:", c + q)   # esperado: soma = 29700

In [0]:
print("tickers distintos:", cotacoes.select("ticker").distinct().count())     # esperado: 450
print("tickers divergentes:", chk.filter(F.col("ticker") != F.col("ticker_cadastro")).count())   # esperado: 0

## Decisões de tratamento
- Colunas constantes ou vazias descartadas (codigo_bdi, tipo_mercado, moeda, fator_cotacao, prazo_dias_termo, codigo_isin).
- Preços e volume: a vírgula é sempre decimal e não há separador de milhar, então a troca por ponto vale para todas as linhas.
- qtd_titulos em bigint, porque pode passar de 2 bilhões.
- Deduplicação por (id_empresa, data_pregao), depois de converter os dois campos.
- Preço de fechamento nulo ou <= 0 vai para silver.cotacoes_quarentena com o motivo, em vez de ser apagado.
- Sem partição: o volume é pequeno e partir criaria arquivos minúsculos.
- JOIN com empresas validado: 0 linhas na bronze (texto) e todas na silver (int).